# Bengali Grapheme-Cluster Unigram Tokenizer

Research pipeline for building and evaluating a grapheme-cluster (akshara) initialized unigram tokenizer for Bengali, with a 2×2 ablation study.

**Steps:** Setup → Build Corpus → EDA → Train → Evaluate

## 1. Setup

Install dependencies and mount/upload your `datasets/` folder. Upload the project or clone from git.

In [ ]:
# Install dependencies
!pip install -q sentencepiece regex pyarrow tiktoken transformers pyyaml tqdm seaborn torch

# If running from a cloned repo:
import os, sys
PROJECT_ROOT = os.getcwd()
if not os.path.exists("src/tokenizer_bn"):
    # Adjust path if notebook is in notebooks/
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("Project root:", PROJECT_ROOT)

# Check GPU availability
from tokenizer_bn.device import device_info
print(device_info("auto"))

In [ ]:
# Mount Google Drive (optional) or upload datasets manually
from google.colab import drive
drive.mount("/content/drive")

# Point to your datasets folder
DATASETS_DIR = "/content/drive/MyDrive/tokenizer/datasets"  # adjust path
# Or upload to /content/datasets and use:
# DATASETS_DIR = "/content/datasets"

## 2. Configuration

In [ ]:
from tokenizer_bn.config import load_config, ensure_dirs
from tokenizer_bn.checkpoint import CheckpointManager
from tokenizer_bn.logging_utils import get_logger
from tokenizer_bn.device import log_device_info, device_info

config = load_config("configs/default.yaml")
# Override datasets path for Colab
config.paths.datasets_dir = __import__("pathlib").Path(DATASETS_DIR)
ensure_dirs(config)
ckpt = CheckpointManager(config)
logger = get_logger("colab", config)
log_device_info(logger, config.device.device)
print(f"Corpus budget: {config.corpus.corpus_sample_bytes / 1e6:.0f} MB")
print(f"Vocab size: {config.training.vocab_size}")
print(f"GPU enabled: {config.device.use_gpu}")
print(device_info(config.device.device))

## 3. Build Processed Bangla Corpus

Extracts Bangla-only text from all datasets, drops English/romanized content, and creates a parallel bn-en eval set.

In [ ]:
from tokenizer_bn.data.build_corpus import build_corpus

if not ckpt.is_step_done("build-corpus"):
    ckpt.mark_step_started("build-corpus")
    manifest = build_corpus(config, ckpt)
    ckpt.mark_step_done("build-corpus", metadata={"corpus_bytes": manifest["corpus_bytes_written"]})
else:
    import json
    with open(config.manifest_path) as f:
        manifest = json.load(f)
    print("Corpus already built (checkpoint).")

print(f"Corpus bytes: {manifest['corpus_bytes_written']:,}")
print(f"Parallel pairs: {manifest['parallel_pairs']:,}")
for src, stats in manifest["sources"].items():
    print(f"  {src}: kept={stats['lines_kept']:,}")

## 4. Exploratory Data Analysis

In [ ]:
from tokenizer_bn.eda.analyze import run_eda
from IPython.display import Image, display
import json

if not ckpt.is_step_done("eda"):
    ckpt.mark_step_started("eda")
    eda_stats = run_eda(config, ckpt)
    ckpt.mark_step_done("eda")
else:
    with open(config.paths.results_dir / "eda" / "eda_stats.json") as f:
        eda_stats = json.load(f)
    print("EDA already done (checkpoint).")

print(json.dumps({k: v for k, v in eda_stats.items() if k != "top_aksharas"}, indent=2))

In [ ]:
# Display EDA charts
from pathlib import Path
eda_dir = config.paths.results_dir / "eda"
for chart in sorted(eda_dir.glob("*.png")):
    print(chart.name)
    display(Image(filename=str(chart)))

## 5. Train 2×2 Ablation Variants

Trains four SentencePiece models: {grapheme, byte} × {unigram, bpe}.

In [ ]:
from tokenizer_bn.train.train_variants import train_all_variants

if not ckpt.is_step_done("train"):
    ckpt.mark_step_started("train")
    train_results = train_all_variants(config, ckpt)
    ckpt.mark_step_done("train", metadata=train_results)
else:
    print("Training already done (checkpoint).")

import os
for variant in ["grapheme_unigram", "grapheme_bpe", "byte_unigram", "byte_bpe"]:
    model_path = config.paths.models_dir / variant / f"{variant}.model"
    if model_path.exists():
        size_mb = os.path.getsize(model_path) / 1e6
        print(f"  {variant}: {size_mb:.1f} MB")

## 6. Evaluate Tokenizers

Computes fertility, chars/token, parity vs English, and STRR. Runs paired statistical tests for RQ1–RQ3.

In [ ]:
from tokenizer_bn.eval.harness import run_evaluation
import pandas as pd

if not ckpt.is_step_done("evaluate"):
    ckpt.mark_step_started("evaluate")
    eval_summary = run_evaluation(config, ckpt)
    ckpt.mark_step_done("evaluate")
else:
    print("Evaluation already done (checkpoint).")

metrics_df = pd.read_csv(config.paths.results_dir / "tables" / "eval_metrics.csv")
corpus_df = pd.read_csv(config.paths.results_dir / "tables" / "corpus_metrics.csv")
print("=== Full Corpus Metrics ===")
display(corpus_df.pivot(index="tokenizer", columns="metric", values="value"))
print("\n=== All Metrics ===")
display(metrics_df.pivot(index="tokenizer", columns="metric", values="value"))

In [ ]:
# Display evaluation charts
fig_dir = config.paths.results_dir / "figures"
for chart in ["corpus_metrics_dashboard.png", "eval_metrics_dashboard.png", "metrics_heatmap.png",
              "corpus_token_count.png", "compression_ratio.png", "tokens_per_bengali_word.png",
              "parity_comparison.png"]:
    path = fig_dir / chart
    if path.exists():
        print(chart)
        display(Image(filename=str(path)))

In [ ]:
# RQ statistical tests
stats_path = config.paths.results_dir / "tables" / "rq_statistical_tests.csv"
if stats_path.exists():
    pd.read_csv(stats_path)
else:
    print("No statistical test results yet.")